# Práctica de Laboratorio 2: DS5345
Este cuaderno contiene la estructura necesaria para completar la práctica de laboratorio 2

In [1]:
!pip install gymnasium numpy matplotlib


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


PARTE I

## Configuración del Entorno

In [ ]:
import gymnasium as gym
import numpy as np
import random

env = gym.make("Taxi-v4")
num_states = env.observation_space.n
num_actions = env.action_space.n
print(env.)
print(f"Estados: {num_states}, Acciones: {num_actions}")

Discrete(500)
Estados: 500, Acciones: 6


## 1. Política Epsilon-Greedy

In [4]:
def epsilon_greedy(Q, state, epsilon):
    if random.uniform(0, 1) < epsilon:
        return env.action_space.sample()
    else:
        return np.argmax(Q[state])

In [7]:
env.reset()

rewards_posibles = set(
    transicion[2]
    for estados in env.unwrapped.P.values()
    for acciones in estados.values()
    for transicion in acciones
)


print("Rewards encontrados en el entorno:", list(rewards_posibles))

env.close()

Rewards encontrados en el entorno: [20, -1, -10]


# Pregunta 1.1

- Existen 500 estados posibles dado que en este particular entorno , las acciones posibles son 6 las cuales son moverse a:
0 : sur
1 : norte 
2 : este
3 : oeste 
4 : recoger pasajero
5 : dejar pasajero

# Pregunta 1.2

Cada paso será -1 , luego de ello si se llega al objetivo de dejar un pasajero serán recompensa de  20 y , finalmente -10 si se ejecuta una acción ilegal de dejar pasajeros y/o recoger pasajeros

## 2. Implementación de SARSA

## Pregunta 1.3

In [ ]:
Q_sarsa = np.zeros([num_states, num_actions])
alpha = 0.1
gamma = 0.99
epsilon = 0.1

for episode in range(1000):
    state, _ = env.reset()
    action = epsilon_greedy(Q_sarsa, state, epsilon)
    done = False
    
    while not done:
        next_state, reward, terminated, truncated, _ = env.step(action)
        next_action = epsilon_greedy(Q_sarsa, next_state, epsilon)
        
        # Implementacion SARSA 

        Q_sarsa[state,action]= Q_sarsa[state , action] + alpha*(reward + gamma*(Q_sarsa[next_state,next_action])-Q_sarsa[state,action])
        
        state, action = next_state, next_action
        done = terminated or truncated

## 3. Implementación de Q-Learning

## Pregunta 1.4

In [ ]:
Q_ql = np.zeros([num_states, num_actions])

for episode in range(1000):
    state, _ = env.reset()
    done = False
    
    while not done:
        action = epsilon_greedy(Q_ql, state, epsilon)
        next_state, reward, terminated, truncated, _ = env.step(action)


        # Implementacion Q Learning

        Q_ql[state , action] = Q_ql[state , action]- alpha*(reward+gamma*(np.max(Q_ql[next_state]))-Q_ql[state,action])
        
        state = next_state
        done = terminated or truncated

## 4. Bonus: Double Q-Learning

In [8]:
Q1 = np.zeros([num_states, num_actions])
Q2 = np.zeros([num_states, num_actions])

for episode in range(1000):
    state, _ = env.reset()
    done = False
    
    while not done:
        action = epsilon_greedy(Q1 + Q2, state, epsilon)
        next_state, reward, terminated, truncated, _ = env.step(action)
        
        if random.uniform(0, 1) < 0.5:
            # Actualizar Q1
            Q1[state , action] = Q1[state , action]- alpha*(reward+gamma*(np.max(Q2[next_state]))-Q1[state,action])
            
        else:
            # Actualizar Q2
            Q2[state , action] = Q2[state , action]- alpha*(reward+gamma*(np.max(Q1[next_state]))-Q2[state,action])
            
            
        state = next_state
        done = terminated or truncated

PARTE 2 VFA CON Q LEARNING

1 Exploracion del entorno

In [10]:
import gymnasium as gym
import numpy as np

env = gym.make("CartPole-v1")
state, _ = env.reset()

print("Estado inicial:", state)
print("Límites superiores:", env.observation_space.high)
print("Límites inferiores:", env.observation_space.low)

Estado inicial: [0.04660726 0.02108078 0.03499522 0.03291357]
Límites superiores: [4.8               inf 0.41887903        inf]
Límites inferiores: [-4.8               -inf -0.41887903        -inf]


# Pregunta 2.1

Las dificultades se encuentran en dos aspectos especificos. En primer lugar en relación al hardware la forma tabular tendría que almancenar una cantidad muy grande de estados Q(s,a) , dado que es un entorno continuo como se observa esta tabla sería una discretización de de esto . La cantidad de estados a almacenar para que el problema pueda ser resuelto en forma tabular es enorme en este caso y provocaría un gran uso de memoría.

Por otro lado, el problema de la probabilidad een que la cantidad de estados continuos es realmente infinita así que con seguridad muchos estados se quedarían sin explorarse así que la optimalidad global del sistema despues del aprendizaje no sería idonea.

Eso por ello que para estos dos casos se utiliza la función de aproximaxión al valor Q(s,a) por medio de la parametrización y un ajuste de curva que trae consigo los features y pesos del modelo . 

2.Aproximacion lineal con funcion valor

In [11]:
# 4 características del estado, 2 acciones posibles
weights = np.zeros((4, 2)) 

def predict_q(state, w):
    # Producto punto para obtener los valores Q de ambas acciones
    return np.dot(state, w)

# Predicción inicial (pesos en cero)
q_values = predict_q(state, weights)
print("Valores Q aproximados:", q_values)

Valores Q aproximados: [0. 0.]


# Pregunta 2.2

Basicamente la ventaje de tener un modelo o curva de ajuste hacia el valor de la forma $Q(s,a) = W^{T}\phi(s)$ es que el modelo carga consigo una aproximaxión (con determinado bias mínimo o error al pasar por el proceso de optimización) al valor no visitado o actualizado, es decir comprime la información continua de los valores Q en todo el espacio de estados y acciones y solo basta evaluar la función para hallar dicho valor , esto es ventajoso , dado que cuando se actualizan los parametros W todo el modelo de ajuste se actualiza sin necesidad de tener que visitar cada uno de estos estados. La ventaja para espacios continuos se encuentra en que no es necesario almacenar una cantidad enorme de información , como en el caso tabular, ya que el modelo o curva de ajuste que aproxima a los valores necesita un par de parametros limitados como los features y pesos W . El tradeoff de esto es que si bien los valores ahora son estimaciones basadas en el modelo con el proceso iterativo la optimización de la función objetivo irá reduciendo el error sobre el valor real. 

3.- Semigradiente con Q learning

In [ ]:
def update_weights_qlearning(state, action, reward, next_state, w, alpha=0.01, gamma=0.99):
    # 1. Predicción actual
    q_current = predict_q(state, w)[action]
    
    # 2. Predicción futura (Target) usando MAX para Q-Learning (Off-policy)
    q_next_max = np.max(predict_q(next_state, w))
    td_target = reward + gamma * q_next_max
    
    # 3. Error TD
    td_error = td_target - q_current
    
    # 4. Actualización (Semi-gradiente)
    # El gradiente de (state * w) respecto a w es simplemente 'state'
    w[:, action] += alpha * td_error * state
    
    return w

# Pregunta 2.3

El problema radica en como el sistema en que Q learning realmente esta resolviendo el problema de optimalizadad de Bellman $Q^{*}(s,a) = E[R_{t+1}+max_{a'}(Q(s',a'))]$ donde para la función objetivo  tendría que ser derivada incluye el valor esperado. Justamente el problema fundamental es ese mismo, la derivada completa exige resolver un valor esperado complicado en ese sentido deriar directamente se hace poco practico. Para resolver este caso se usa una aproximación estocastica del descenso del gradiente que que puede asumir como se ve en el código que el valor W respecto a un estado es fijo y no se tiene que derivar en base a ese parametro en su totalidad. Es decir tratas el target $R_{t+1}+max_{a'}(Q(s',a'))$ como fijo en una iteración especifica y solo derivas la estimación actual.

# Pregunta 2.4

Los tres elementos principales son:
- off policy learning : Aprende sobre un tipo de politica determinado mientras usa la experiencia de un tipo de un tipo de poltica diferente , por ejemplo usando una $\epsilon - greedy$
- bootstrapping : En este caso haces las actualizaciones basandose en estaimaciones propias del target e ignora la recompensa final del episodio.
- Aproximación de funciones : Cuando los parametros de la red se actualizan estos alteran todo el estado de la red independientemente del estado objetivo que quería optimizar provocando que otros valores tambien se actualicen

En suma con todos estos elementos primero se crea un bootstrapping inicial con un target que introduce errores , luego estos errores se esparcen por todo el sistema en general por medio de la generalización que se da al actualizar parametros de la red neuronal. Como el sistema es off policy no se garantiza que se optimicen los estados necesarios para corregir esos errores amplifica pesos y dispara la red de una manera errada, en suma, todo lo anterior generaría un bucle de error que crece y se retroalimenta a si mismo.